In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

print("pandas version:", pd.__version__)
print("numpy version:", np.__version__)
print("Packages loaded successfully.")

In [ ]:
from pathlib import Path

project_folder = Path.cwd()
data_raw = project_folder / "data_raw"
data_clean = project_folder / "data_clean"
outputs = project_folder / "outputs"

print("Current project folder:", project_folder)
print("Raw data folder exists:", data_raw.exists())
print("Clean data folder exists:", data_clean.exists())
print("Outputs folder exists:", outputs.exists())

print("\nFiles in data_raw:")
for file in data_raw.iterdir():
    print("-", file.name)

In [ ]:
datasets = {}

for file in data_raw.iterdir():
    file_name = file.stem.lower().replace(" ", "_")
    
    if file.suffix.lower() == ".csv":
        datasets[file_name] = pd.read_csv(file)
        print(f"Loaded CSV: {file.name}")
        
    elif file.suffix.lower() in [".xlsx", ".xls"]:
        datasets[file_name] = pd.read_excel(file)
        print(f"Loaded Excel: {file.name}")
        
    else:
        print(f"Skipped unsupported file: {file.name}")

print("\nDatasets loaded:")
print(list(datasets.keys()))

In [ ]:
for name, df in datasets.items():
    print("=" * 80)
    print(f"Dataset name: {name}")
    print(f"Shape: {df.shape}")
    display(df.head())

In [ ]:
from pathlib import Path
import urllib.request

# Set project folders
project_folder = Path.cwd()
data_raw = project_folder / "data_raw"
data_raw.mkdir(exist_ok=True)

# Dataset download links
downloads = {
    # Main geopolitical risk index from Caldara & Iacoviello
    "gpr_index.xls": "https://www.matteoiacoviello.com/gpr_files/data_gpr_export.xls",
    
    # FRED datasets
    "vix.csv": "https://fred.stlouisfed.org/graph/fredgraph.csv?id=VIXCLS",
    "oil_wti.csv": "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DCOILWTICO",
    "cpi.csv": "https://fred.stlouisfed.org/graph/fredgraph.csv?id=CPIAUCSL",
    "fed_funds_rate.csv": "https://fred.stlouisfed.org/graph/fredgraph.csv?id=FEDFUNDS"
}

# Download files
for filename, url in downloads.items():
    filepath = data_raw / filename
    
    print(f"Downloading {filename}...")
    urllib.request.urlretrieve(url, filepath)
    print(f"Saved to {filepath}")

print("\nAll downloads complete.")
print("\nFiles now in data_raw:")

for file in data_raw.iterdir():
    print("-", file.name)

In [ ]:
from pathlib import Path
import urllib.request

# Set project folders
project_folder = Path.cwd()
data_raw = project_folder / "data_raw"
data_raw.mkdir(exist_ok=True)

# Dataset download links
downloads = {
    "gpr_index.xls": "https://www.matteoiacoviello.com/gpr_files/data_gpr_export.xls",
    "vix.csv": "https://fred.stlouisfed.org/graph/fredgraph.csv?id=VIXCLS",
    "oil_wti.csv": "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DCOILWTICO",
    "oil_brent.csv": "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DCOILBRENTEU",
    "cpi.csv": "https://fred.stlouisfed.org/graph/fredgraph.csv?id=CPIAUCSL",
    "fed_funds_rate.csv": "https://fred.stlouisfed.org/graph/fredgraph.csv?id=FEDFUNDS"
}

# Download files
for filename, url in downloads.items():
    filepath = data_raw / filename
    
    print(f"Downloading {filename}...")
    urllib.request.urlretrieve(url, filepath)
    print(f"Saved to {filepath}")

print("\nAll downloads complete.")
print("\nFiles now in data_raw:")

for file in data_raw.iterdir():
    print("-", file.name)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from pathlib import Path

project_folder = Path.cwd()
data_raw = project_folder / "data_raw"

datasets = {}

for file in data_raw.iterdir():
    file_name = file.stem.lower().replace(" ", "_")
    
    if file.suffix.lower() == ".csv":
        datasets[file_name] = pd.read_csv(file)
        print(f"Loaded CSV: {file.name}")
        
    elif file.suffix.lower() in [".xlsx", ".xls"]:
        datasets[file_name] = pd.read_excel(file)
        print(f"Loaded Excel: {file.name}")
        
    else:
        print(f"Skipped unsupported file: {file.name}")

print("\nDatasets loaded:")
print(list(datasets.keys()))

In [ ]:
for name, df in datasets.items():
    print("=" * 80)
    print(f"Dataset name: {name}")
    print(f"Shape: {df.shape}")
    display(df.head())

In [ ]:
from pathlib import Path
import pandas as pd

project_folder = Path.cwd()
data_raw = project_folder / "data_raw"

print("Files in data_raw:")
for file in data_raw.iterdir():
    print("-", file.name)

In [ ]:
datasets = {}

for file in data_raw.iterdir():
    file_name = file.stem.lower().replace(" ", "_")
    
    if file.suffix.lower() == ".csv":
        datasets[file_name] = pd.read_csv(file)
        print(f"Loaded CSV: {file.name}")
        
    elif file.suffix.lower() in [".xlsx", ".xls"]:
        datasets[file_name] = pd.read_excel(file)
        print(f"Loaded Excel: {file.name}")
        
    else:
        print(f"Skipped unsupported file: {file.name}")

print("\nDatasets loaded:")
print(list(datasets.keys()))

In [ ]:
for name, df in datasets.items():
    print("=" * 80)
    print(name)
    print(df.shape)
    print(df.columns.tolist())
    display(df.head())

In [ ]:
# Create clean copies of each dataset
cpi = datasets["cpi"].copy()
fed = datasets["fed_funds_rate"].copy()
gpr = datasets["gpr_index"].copy()
brent = datasets["oil_brent"].copy()
wti = datasets["oil_wti"].copy()
vix = datasets["vix"].copy()

# Function to clean FRED daily/monthly files
def clean_fred_monthly(df, date_col, value_col, new_name):
    temp = df[[date_col, value_col]].copy()
    temp[date_col] = pd.to_datetime(temp[date_col])
    temp[value_col] = pd.to_numeric(temp[value_col], errors="coerce")
    
    # Convert to monthly average
    temp = (
        temp.set_index(date_col)
            .resample("ME")
            .mean()
            .reset_index()
    )
    
    temp = temp.rename(columns={
        date_col: "date",
        value_col: new_name
    })
    
    return temp

# Clean financial and economic variables
vix_m = clean_fred_monthly(vix, "observation_date", "VIXCLS", "vix")
brent_m = clean_fred_monthly(brent, "observation_date", "DCOILBRENTEU", "brent")
wti_m = clean_fred_monthly(wti, "observation_date", "DCOILWTICO", "wti")
cpi_m = clean_fred_monthly(cpi, "observation_date", "CPIAUCSL", "cpi")
fed_m = clean_fred_monthly(fed, "observation_date", "FEDFUNDS", "fed_funds_rate")

# Clean GPR monthly data
gpr_m = gpr[[
    "month",
    "GPR",
    "GPRC_USA",
    "GPRC_CHN",
    "GPRC_RUS"
]].copy()

gpr_m["month"] = pd.to_datetime(gpr_m["month"])

gpr_m = gpr_m.rename(columns={
    "month": "date",
    "GPR": "gpr_global",
    "GPRC_USA": "gpr_usa",
    "GPRC_CHN": "gpr_china",
    "GPRC_RUS": "gpr_russia"
})

# Make date format consistent: month end
gpr_m["date"] = gpr_m["date"] + pd.offsets.MonthEnd(0)

# Merge all monthly datasets
merged = (
    gpr_m
    .merge(vix_m, on="date", how="inner")
    .merge(brent_m, on="date", how="inner")
    .merge(wti_m, on="date", how="inner")
    .merge(cpi_m, on="date", how="inner")
    .merge(fed_m, on="date", how="inner")
)

# Restrict to 2010 through 2025
merged = merged[(merged["date"] >= "2010-01-01") & (merged["date"] <= "2025-12-31")].copy()

# Create monthly inflation rate from CPI
merged["inflation_rate"] = merged["cpi"].pct_change(12) * 100

# Drop missing rows caused by inflation calculation
merged = merged.dropna().reset_index(drop=True)

# Preview merged data
print("Merged dataset shape:", merged.shape)
display(merged.head())
display(merged.tail())

In [ ]:
print(merged.info())

print("\nSummary statistics:")
display(merged.describe())

In [ ]:
data_clean = Path.cwd() / "data_clean"
data_clean.mkdir(exist_ok=True)

merged.to_excel(data_clean / "merged_monthly_data.xlsx", index=False)
merged.to_csv(data_clean / "merged_monthly_data.csv", index=False)

print("Cleaned dataset saved to data_clean folder.")

In [ ]:
import statsmodels.api as sm

# Main regression model: Brent as the oil control
y = merged["vix"]

X = merged[[
    "gpr_global",
    "brent",
    "inflation_rate",
    "fed_funds_rate"
]]

X = sm.add_constant(X)

model_brent = sm.OLS(y, X).fit()

print(model_brent.summary())

In [ ]:
# Robustness/comparison model: WTI as the oil control
y = merged["vix"]

X = merged[[
    "gpr_global",
    "wti",
    "inflation_rate",
    "fed_funds_rate"
]]

X = sm.add_constant(X)

model_wti = sm.OLS(y, X).fit()

print(model_wti.summary())

In [ ]:
comparison = pd.DataFrame({
    "Brent Model Coefficient": model_brent.params,
    "Brent Model P-Value": model_brent.pvalues,
    "WTI Model Coefficient": model_wti.params,
    "WTI Model P-Value": model_wti.pvalues
})

display(comparison)

In [ ]:
# Country-specific geopolitical risk model using Brent

y = merged["vix"]

X = merged[[
    "gpr_usa",
    "gpr_china",
    "gpr_russia",
    "brent",
    "inflation_rate",
    "fed_funds_rate"
]]

X = sm.add_constant(X)

model_country_brent = sm.OLS(y, X).fit()

print(model_country_brent.summary())

In [ ]:
# Country-specific geopolitical risk model using WTI

y = merged["vix"]

X = merged[[
    "gpr_usa",
    "gpr_china",
    "gpr_russia",
    "wti",
    "inflation_rate",
    "fed_funds_rate"
]]

X = sm.add_constant(X)

model_country_wti = sm.OLS(y, X).fit()

print(model_country_wti.summary())

In [ ]:
# Create one-month lagged country-specific GPR variables
merged["gpr_usa_lag1"] = merged["gpr_usa"].shift(1)
merged["gpr_china_lag1"] = merged["gpr_china"].shift(1)
merged["gpr_russia_lag1"] = merged["gpr_russia"].shift(1)

# Drop rows with missing lag values
merged_lag = merged.dropna().copy()

# Lagged country-specific model using Brent
y = merged_lag["vix"]

X = merged_lag[[
    "gpr_usa_lag1",
    "gpr_china_lag1",
    "gpr_russia_lag1",
    "brent",
    "inflation_rate",
    "fed_funds_rate"
]]

X = sm.add_constant(X)

model_lag_brent = sm.OLS(y, X).fit()

print(model_lag_brent.summary())

In [ ]:
# Lagged country-specific model using WTI
y = merged_lag["vix"]

X = merged_lag[[
    "gpr_usa_lag1",
    "gpr_china_lag1",
    "gpr_russia_lag1",
    "wti",
    "inflation_rate",
    "fed_funds_rate"
]]

X = sm.add_constant(X)

model_lag_wti = sm.OLS(y, X).fit()

print(model_lag_wti.summary())

In [ ]:
# Create clean comparison table for all key models

results_table = pd.DataFrame({
    "Global GPR + Brent": [
        model_brent.params.get("gpr_global"),
        model_brent.pvalues.get("gpr_global"),
        model_brent.rsquared,
        model_brent.rsquared_adj,
        int(model_brent.nobs)
    ],
    "Global GPR + WTI": [
        model_wti.params.get("gpr_global"),
        model_wti.pvalues.get("gpr_global"),
        model_wti.rsquared,
        model_wti.rsquared_adj,
        int(model_wti.nobs)
    ],
    "Country GPR + Brent": [
        model_country_brent.params.get("gpr_usa"),
        model_country_brent.pvalues.get("gpr_usa"),
        model_country_brent.rsquared,
        model_country_brent.rsquared_adj,
        int(model_country_brent.nobs)
    ],
    "Country GPR + WTI": [
        model_country_wti.params.get("gpr_usa"),
        model_country_wti.pvalues.get("gpr_usa"),
        model_country_wti.rsquared,
        model_country_wti.rsquared_adj,
        int(model_country_wti.nobs)
    ],
    "Lagged Country GPR + Brent": [
        model_lag_brent.params.get("gpr_usa_lag1"),
        model_lag_brent.pvalues.get("gpr_usa_lag1"),
        model_lag_brent.rsquared,
        model_lag_brent.rsquared_adj,
        int(model_lag_brent.nobs)
    ],
    "Lagged Country GPR + WTI": [
        model_lag_wti.params.get("gpr_usa_lag1"),
        model_lag_wti.pvalues.get("gpr_usa_lag1"),
        model_lag_wti.rsquared,
        model_lag_wti.rsquared_adj,
        int(model_lag_wti.nobs)
    ]
}, index=[
    "Key GPR Coefficient",
    "Key GPR P-Value",
    "R-Squared",
    "Adjusted R-Squared",
    "Observations"
])

display(results_table)

In [ ]:
outputs = Path.cwd() / "outputs"
outputs.mkdir(exist_ok=True)

results_table.to_excel(outputs / "regression_results_summary.xlsx")

print("Regression results saved to outputs folder.")

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

# Make sure data is sorted
merged = merged.sort_values("date").reset_index(drop=True)

# Create charts folder
charts_folder = Path.cwd() / "outputs" / "charts"
charts_folder.mkdir(parents=True, exist_ok=True)

print("Charts will be saved in:", charts_folder)

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(merged["date"], merged["vix"])
plt.title("VIX Over Time (2010–2025)")
plt.xlabel("Date")
plt.ylabel("VIX")
plt.grid(True)
plt.tight_layout()
plt.savefig(charts_folder / "vix_over_time.png", dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(merged["date"], merged["gpr_global"])
plt.title("Global Geopolitical Risk Index Over Time (2010–2025)")
plt.xlabel("Date")
plt.ylabel("Global GPR")
plt.grid(True)
plt.tight_layout()
plt.savefig(charts_folder / "global_gpr_over_time.png", dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(merged["date"], merged["gpr_usa"], label="U.S. GPR")
plt.plot(merged["date"], merged["gpr_china"], label="China GPR")
plt.plot(merged["date"], merged["gpr_russia"], label="Russia GPR")
plt.title("Country-Specific Geopolitical Risk Indices (2010–2025)")
plt.xlabel("Date")
plt.ylabel("Country GPR")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(charts_folder / "country_gpr_over_time.png", dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(merged["date"], merged["brent"], label="Brent")
plt.plot(merged["date"], merged["wti"], label="WTI")
plt.title("Oil Prices Over Time (2010–2025)")
plt.xlabel("Date")
plt.ylabel("Oil Price")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(charts_folder / "oil_prices_over_time.png", dpi=300)
plt.show()

In [ ]:
corr_vars = [
    "vix",
    "gpr_global",
    "gpr_usa",
    "gpr_china",
    "gpr_russia",
    "brent",
    "wti",
    "inflation_rate",
    "fed_funds_rate"
]

corr_matrix = merged[corr_vars].corr()

plt.figure(figsize=(10, 8))
plt.imshow(corr_matrix, aspect="auto")
plt.colorbar()
plt.xticks(range(len(corr_vars)), corr_vars, rotation=45, ha="right")
plt.yticks(range(len(corr_vars)), corr_vars)
plt.title("Correlation Heatmap")

for i in range(len(corr_vars)):
    for j in range(len(corr_vars)):
        plt.text(j, i, f"{corr_matrix.iloc[i, j]:.2f}", ha="center", va="center")

plt.tight_layout()
plt.savefig(charts_folder / "correlation_heatmap.png", dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(merged_lag["gpr_usa_lag1"], merged_lag["vix"])

# Add fitted line
x = merged_lag["gpr_usa_lag1"]
y = merged_lag["vix"]
m, b = np.polyfit(x, y, 1)
plt.plot(x, m * x + b)

plt.title("Lagged U.S. GPR vs VIX")
plt.xlabel("Lagged U.S. GPR")
plt.ylabel("VIX")
plt.grid(True)
plt.tight_layout()
plt.savefig(charts_folder / "lagged_us_gpr_vs_vix.png", dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(merged["date"], merged["inflation_rate"], label="Inflation Rate")
plt.plot(merged["date"], merged["fed_funds_rate"], label="Fed Funds Rate")
plt.title("Inflation and Federal Funds Rate Over Time (2010–2025)")
plt.xlabel("Date")
plt.ylabel("Percent")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(charts_folder / "inflation_and_fedfunds_over_time.png", dpi=300)
plt.show()

In [ ]:
print("Saved chart files:")
for file in charts_folder.iterdir():
    print("-", file.name)

In [ ]:
# ------------------------------------------------------------
# Enhanced model variables
# ------------------------------------------------------------

# Make sure data is sorted by date
merged = merged.sort_values("date").reset_index(drop=True)

# Lagged dependent variable
merged["vix_lag1"] = merged["vix"].shift(1)

# Lagged country-specific geopolitical risk variables
merged["gpr_usa_lag1"] = merged["gpr_usa"].shift(1)
merged["gpr_china_lag1"] = merged["gpr_china"].shift(1)
merged["gpr_russia_lag1"] = merged["gpr_russia"].shift(1)

# Crisis-period dummy variables
merged["covid_period"] = (
    (merged["date"] >= "2020-03-01") & 
    (merged["date"] <= "2020-12-31")
).astype(int)

merged["ukraine_war_period"] = (
    (merged["date"] >= "2022-02-01") & 
    (merged["date"] <= "2022-12-31")
).astype(int)

# Drop rows with missing lagged values
merged_enhanced = merged.dropna().copy()

print("Enhanced dataset shape:", merged_enhanced.shape)

display(
    merged_enhanced[
        [
            "date",
            "vix",
            "vix_lag1",
            "gpr_usa_lag1",
            "gpr_china_lag1",
            "gpr_russia_lag1",
            "brent",
            "wti",
            "inflation_rate",
            "fed_funds_rate",
            "covid_period",
            "ukraine_war_period",
        ]
    ].head()
)

In [ ]:
# ------------------------------------------------------------
# Enhanced Brent model
# ------------------------------------------------------------

y = merged_enhanced["vix"]

X = merged_enhanced[
    [
        "vix_lag1",
        "gpr_usa_lag1",
        "gpr_china_lag1",
        "gpr_russia_lag1",
        "brent",
        "inflation_rate",
        "fed_funds_rate",
        "covid_period",
        "ukraine_war_period",
    ]
]

X = sm.add_constant(X)

model_enhanced_brent = sm.OLS(y, X).fit()

print(model_enhanced_brent.summary())

In [ ]:
# ------------------------------------------------------------
# Enhanced WTI model
# ------------------------------------------------------------

y = merged_enhanced["vix"]

X = merged_enhanced[
    [
        "vix_lag1",
        "gpr_usa_lag1",
        "gpr_china_lag1",
        "gpr_russia_lag1",
        "wti",
        "inflation_rate",
        "fed_funds_rate",
        "covid_period",
        "ukraine_war_period",
    ]
]

X = sm.add_constant(X)

model_enhanced_wti = sm.OLS(y, X).fit()

print(model_enhanced_wti.summary())

In [ ]:
# ------------------------------------------------------------
# Updated regression comparison table
# ------------------------------------------------------------

results_table_updated = pd.DataFrame({
    "Global GPR + Brent": [
        model_brent.params.get("gpr_global"),
        model_brent.pvalues.get("gpr_global"),
        model_brent.rsquared,
        model_brent.rsquared_adj,
        int(model_brent.nobs)
    ],
    "Global GPR + WTI": [
        model_wti.params.get("gpr_global"),
        model_wti.pvalues.get("gpr_global"),
        model_wti.rsquared,
        model_wti.rsquared_adj,
        int(model_wti.nobs)
    ],
    "Country GPR + Brent": [
        model_country_brent.params.get("gpr_usa"),
        model_country_brent.pvalues.get("gpr_usa"),
        model_country_brent.rsquared,
        model_country_brent.rsquared_adj,
        int(model_country_brent.nobs)
    ],
    "Country GPR + WTI": [
        model_country_wti.params.get("gpr_usa"),
        model_country_wti.pvalues.get("gpr_usa"),
        model_country_wti.rsquared,
        model_country_wti.rsquared_adj,
        int(model_country_wti.nobs)
    ],
    "Lagged Country GPR + Brent": [
        model_lag_brent.params.get("gpr_usa_lag1"),
        model_lag_brent.pvalues.get("gpr_usa_lag1"),
        model_lag_brent.rsquared,
        model_lag_brent.rsquared_adj,
        int(model_lag_brent.nobs)
    ],
    "Lagged Country GPR + WTI": [
        model_lag_wti.params.get("gpr_usa_lag1"),
        model_lag_wti.pvalues.get("gpr_usa_lag1"),
        model_lag_wti.rsquared,
        model_lag_wti.rsquared_adj,
        int(model_lag_wti.nobs)
    ],
    "Enhanced Model + Brent": [
        model_enhanced_brent.params.get("gpr_usa_lag1"),
        model_enhanced_brent.pvalues.get("gpr_usa_lag1"),
        model_enhanced_brent.rsquared,
        model_enhanced_brent.rsquared_adj,
        int(model_enhanced_brent.nobs)
    ],
    "Enhanced Model + WTI": [
        model_enhanced_wti.params.get("gpr_usa_lag1"),
        model_enhanced_wti.pvalues.get("gpr_usa_lag1"),
        model_enhanced_wti.rsquared,
        model_enhanced_wti.rsquared_adj,
        int(model_enhanced_wti.nobs)
    ],
}, index=[
    "Key GPR Coefficient",
    "Key GPR P-Value",
    "R-Squared",
    "Adjusted R-Squared",
    "Observations"
])

display(results_table_updated)

In [ ]:
outputs = Path.cwd() / "outputs"
outputs.mkdir(exist_ok=True)

results_table_updated.to_excel(outputs / "regression_results_summary.xlsx")

print("Updated regression results saved to outputs/regression_results_summary.xlsx")

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(merged_enhanced["date"], merged_enhanced["vix"], label="Current VIX")
plt.plot(merged_enhanced["date"], merged_enhanced["vix_lag1"], label="Lagged VIX")
plt.title("Current VIX and One-Month Lagged VIX Over Time")
plt.xlabel("Date")
plt.ylabel("VIX")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(charts_folder / "vix_and_lagged_vix_over_time.png", dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(merged_enhanced["date"], merged_enhanced["vix"], label="VIX")

# Highlight COVID period
plt.axvspan(
    pd.to_datetime("2020-03-01"),
    pd.to_datetime("2020-12-31"),
    alpha=0.2,
    label="COVID Period"
)

# Highlight Ukraine war period
plt.axvspan(
    pd.to_datetime("2022-02-01"),
    pd.to_datetime("2022-12-31"),
    alpha=0.2,
    label="Ukraine War Period"
)

plt.title("VIX with COVID and Ukraine War Periods Highlighted")
plt.xlabel("Date")
plt.ylabel("VIX")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(charts_folder / "vix_crisis_periods_highlighted.png", dpi=300)
plt.show()

In [ ]:
# ------------------------------------------------------------
# Forecasting dataset
# ------------------------------------------------------------

forecast_df = merged_enhanced.copy()

# Keep only the variables needed for forecasting
forecast_vars = [
    "date",
    "vix",
    "vix_lag1",
    "gpr_usa_lag1",
    "gpr_china_lag1",
    "gpr_russia_lag1",
    "brent",
    "wti",
    "inflation_rate",
    "fed_funds_rate",
    "covid_period",
    "ukraine_war_period"
]

forecast_df = forecast_df[forecast_vars].dropna().copy()

print("Forecasting dataset shape:", forecast_df.shape)
display(forecast_df.head())
display(forecast_df.tail())

In [ ]:
# ------------------------------------------------------------
# Train/test split
# ------------------------------------------------------------

train = forecast_df[forecast_df["date"] < "2023-01-01"].copy()
test = forecast_df[forecast_df["date"] >= "2023-01-01"].copy()

print("Training period:")
print(train["date"].min(), "to", train["date"].max())
print("Training observations:", len(train))

print("\nTesting period:")
print(test["date"].min(), "to", test["date"].max())
print("Testing observations:", len(test))

In [ ]:
# ------------------------------------------------------------
# Baseline forecast model: VIX predicted by lagged VIX only
# ------------------------------------------------------------

y_train = train["vix"]
X_train_baseline = train[["vix_lag1"]]
X_train_baseline = sm.add_constant(X_train_baseline)

baseline_forecast_model = sm.OLS(y_train, X_train_baseline).fit()

print(baseline_forecast_model.summary())

In [ ]:
# Predict test-period VIX using baseline model

X_test_baseline = test[["vix_lag1"]]
X_test_baseline = sm.add_constant(X_test_baseline)

test["baseline_predicted_vix"] = baseline_forecast_model.predict(X_test_baseline)

display(test[["date", "vix", "baseline_predicted_vix"]].head())

In [ ]:
# ------------------------------------------------------------
# Enhanced forecast model
# ------------------------------------------------------------

enhanced_features_brent = [
    "vix_lag1",
    "gpr_usa_lag1",
    "gpr_china_lag1",
    "gpr_russia_lag1",
    "brent",
    "inflation_rate",
    "fed_funds_rate",
    "covid_period",
    "ukraine_war_period"
]

y_train = train["vix"]
X_train_enhanced = train[enhanced_features_brent]
X_train_enhanced = sm.add_constant(X_train_enhanced)

enhanced_forecast_model = sm.OLS(y_train, X_train_enhanced).fit()

print(enhanced_forecast_model.summary())

In [ ]:
# Predict test-period VIX using enhanced model

X_test_enhanced = test[enhanced_features_brent]
X_test_enhanced = sm.add_constant(X_test_enhanced)

test["enhanced_predicted_vix"] = enhanced_forecast_model.predict(X_test_enhanced)

display(test[["date", "vix", "baseline_predicted_vix", "enhanced_predicted_vix"]].head())

In [ ]:
# ------------------------------------------------------------
# Forecast accuracy metrics
# ------------------------------------------------------------

def forecast_metrics(actual, predicted):
    actual = np.array(actual)
    predicted = np.array(predicted)

    mae = np.mean(np.abs(actual - predicted))
    rmse = np.sqrt(np.mean((actual - predicted) ** 2))
    mape = np.mean(np.abs((actual - predicted) / actual)) * 100

    return mae, rmse, mape


baseline_mae, baseline_rmse, baseline_mape = forecast_metrics(
    test["vix"],
    test["baseline_predicted_vix"]
)

enhanced_mae, enhanced_rmse, enhanced_mape = forecast_metrics(
    test["vix"],
    test["enhanced_predicted_vix"]
)

forecast_accuracy = pd.DataFrame({
    "Model": ["Baseline: Lagged VIX Only", "Enhanced: GPR + Macro + Crisis Controls"],
    "MAE": [baseline_mae, enhanced_mae],
    "RMSE": [baseline_rmse, enhanced_rmse],
    "MAPE (%)": [baseline_mape, enhanced_mape]
})

forecast_accuracy = forecast_accuracy.round(4)

display(forecast_accuracy)

In [ ]:
# ------------------------------------------------------------
# Actual vs predicted VIX forecast chart
# ------------------------------------------------------------

plt.figure(figsize=(12, 6))

plt.plot(test["date"], test["vix"], label="Actual VIX")
plt.plot(test["date"], test["baseline_predicted_vix"], label="Baseline Predicted VIX")
plt.plot(test["date"], test["enhanced_predicted_vix"], label="Enhanced Predicted VIX")

plt.title("Actual vs Predicted VIX Forecasts, 2023-2025")
plt.xlabel("Date")
plt.ylabel("VIX")
plt.legend()
plt.grid(True)
plt.tight_layout()

plt.savefig(charts_folder / "actual_vs_predicted_vix_forecast.png", dpi=300)
plt.show()

In [ ]:
# ------------------------------------------------------------
# Forecast errors over time
# ------------------------------------------------------------

test["baseline_error"] = test["vix"] - test["baseline_predicted_vix"]
test["enhanced_error"] = test["vix"] - test["enhanced_predicted_vix"]

plt.figure(figsize=(12, 6))

plt.plot(test["date"], test["baseline_error"], label="Baseline Forecast Error")
plt.plot(test["date"], test["enhanced_error"], label="Enhanced Forecast Error")
plt.axhline(0, linestyle="--")

plt.title("Forecast Errors Over Time, 2023-2025")
plt.xlabel("Date")
plt.ylabel("Actual VIX - Predicted VIX")
plt.legend()
plt.grid(True)
plt.tight_layout()

plt.savefig(charts_folder / "forecast_errors_over_time.png", dpi=300)
plt.show()

In [ ]:
# ------------------------------------------------------------
# Save forecast outputs
# ------------------------------------------------------------

outputs = Path.cwd() / "outputs"
outputs.mkdir(exist_ok=True)

forecast_output = test[
    [
        "date",
        "vix",
        "baseline_predicted_vix",
        "enhanced_predicted_vix",
        "baseline_error",
        "enhanced_error"
    ]
].copy()

forecast_output.to_excel(outputs / "forecast_backtest_results.xlsx", index=False)
forecast_output.to_csv(outputs / "forecast_backtest_results.csv", index=False)

forecast_accuracy.to_excel(outputs / "forecast_accuracy_summary.xlsx", index=False)

print("Forecast backtest results saved:")
print("-", outputs / "forecast_backtest_results.xlsx")
print("-", outputs / "forecast_accuracy_summary.xlsx")

In [ ]:
print("Updated chart files:")
for file in charts_folder.iterdir():
    print("-", file.name)